# YOLO11n-seg + SimAM+CA + Boundary-Aware Mask Loss (Nhóm B — B2)

**Cải tiến:** Thay thế BCE Loss mặc định cho masks bằng BCE + Boundary IoU Loss.  
**Lý do:** BCE Loss đánh giá đều mọi pixel mà không phân biệt pixel biên giới (quan trọng) và pixel lõi (ít quan trọng). Boundary-Aware Loss cung cấp gradient trực tiếp cho vùng biên giới mask → mask boundary sắc nét hơn.  
**Kỳ vọng:** mAP50-95 tăng từ ~0.17 lên 0.22-0.28 nhờ gradient rõ ràng tại ranh giới BG/WSSV.  
**Base model:** SimAM+CA (best model từ notebook combined).


In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec('roboflow') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'])

if importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])

print('Dependencies ready.')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
import os
from roboflow import Roboflow

ROBOFLOW_API_KEY_DIRECT = 'KOEk0qLzBFDc7zfyxtgs'
ROBOFLOW_WORKSPACE    = 'lets-try-this'
ROBOFLOW_PROJECT      = 'shrimpdishandsegv2'
ROBOFLOW_VERSION      = 1
ROBOFLOW_FORMAT       = 'yolo26'


def get_roboflow_api_key():
    if ROBOFLOW_API_KEY_DIRECT.strip():
        return ROBOFLOW_API_KEY_DIRECT.strip()
    try:
        from google.colab import userdata
        return userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        pass
    return os.environ.get('ROBOFLOW_API_KEY', '').strip()


api_key = get_roboflow_api_key()
rf      = Roboflow(api_key=api_key)
dataset = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT).version(ROBOFLOW_VERSION).download(ROBOFLOW_FORMAT)

base_path = '/content/shrimpDisHandSegV2-1' if os.path.exists('/content') else '/kaggle/working/shrimpDisHandSegV2-1'
print('Dataset root:', base_path)

In [ ]:
# ── Grouped-stratified split (anti-leakage) — identical to baseline ────────
import re, shutil, random
from collections import defaultdict, Counter
from pathlib import Path

SEED = 42
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
TRAIN_RATIO, VAL_RATIO = 0.80, 0.10
SHRIMP_NAME_PATTERN = re.compile(
    r'^(?P<disease>Healthy|BG|WSSV_BG|WSSV)-(?P<shrimp_id>.+)-img-(?P<img_num>\d+)$',
    re.IGNORECASE,
)

def normalize_roboflow_stem(stem):
    stem = re.sub(r'_(jpg|jpeg|png|bmp|webp)\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    return re.sub(r'\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)

def parse_shrimp_group_key(image_name):
    stem  = normalize_roboflow_stem(Path(image_name).stem)
    match = SHRIMP_NAME_PATTERN.match(stem)
    if not match:
        return f'unparsed::{stem}', 'unparsed', None, None
    d = match.group('disease')
    s = match.group('shrimp_id')
    return f'{d.lower()}::{s}', d, s, int(match.group('img_num'))

def image_files_in_split(split):
    d = Path(base_path) / split / 'images'
    return sorted(p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

def move_image_and_label(image_path, target_split):
    ti = Path(base_path) / target_split / 'images'; ti.mkdir(parents=True, exist_ok=True)
    tl = Path(base_path) / target_split / 'labels'; tl.mkdir(parents=True, exist_ok=True)
    ls = image_path.parent.parent / 'labels' / f'{image_path.stem}.txt'
    id_ = ti / image_path.name; ld = tl / f'{image_path.stem}.txt'
    if image_path.resolve() != id_.resolve(): shutil.move(str(image_path), str(id_))
    if ls.exists():
        if ls.resolve() != ld.resolve(): shutil.move(str(ls), str(ld))
    else:
        ld.write_text('')

def disease_for_group(filenames):
    return Counter(parse_shrimp_group_key(f)[1] for f in filenames).most_common(1)[0][0]

def split_one_stratum(items):
    n = len(items); tr = int(TRAIN_RATIO*n); va = int(VAL_RATIO*n); te = n-tr-va
    if n >= 3:
        if va == 0: va, tr = 1, tr-1
        if te == 0: te, tr = 1, tr-1
    if tr < 1 and n > 0: tr = 1
    while tr+va+te > n: tr -= 1
    return items[:tr], items[tr:tr+va], items[tr+va:]

def remove_yolo_label_caches(root):
    for p in Path(root).glob('**/*.cache'): p.unlink()

for split in ['train', 'valid', 'test']:
    for sub in ['images', 'labels']:
        os.makedirs(os.path.join(base_path, split, sub), exist_ok=True)

all_images = []
for split in ['train', 'valid', 'test']:
    all_images.extend(image_files_in_split(split))
for img in sorted(all_images):
    move_image_and_label(img, 'train')

groups = defaultdict(list)
for img in image_files_in_split('train'):
    gk, *_ = parse_shrimp_group_key(img.name)
    groups[gk].append(img.name)

strata = defaultdict(list)
for gk, fn in sorted(groups.items()):
    strata[disease_for_group(fn)].append((gk, fn))

split_to_groups = {'train': [], 'valid': [], 'test': []}
rng = random.Random(SEED)
for disease, items in sorted(strata.items()):
    items = sorted(items, key=lambda x: x[0]); rng.shuffle(items)
    tr, va, te = split_one_stratum(items)
    split_to_groups['train'].extend(tr)
    split_to_groups['valid'].extend(va)
    split_to_groups['test'].extend(te)
    print(f'  {disease}: {len(tr)} train, {len(va)} valid, {len(te)} test groups')

for split, sg in split_to_groups.items():
    for _, fns in sg:
        for fn in fns:
            move_image_and_label(Path(base_path)/'train'/'images'/fn, split)

g2s = {}
for split in ['train', 'valid', 'test']:
    for ip in image_files_in_split(split):
        gk, *_ = parse_shrimp_group_key(ip.name)
        prev = g2s.setdefault(gk, split)
        if prev != split:
            raise RuntimeError(f'Leakage: {gk} in {prev} and {split}')
print('Leakage check PASSED.')
remove_yolo_label_caches(base_path)
for split in ['train', 'valid', 'test']:
    print(f'  {split}: {len(image_files_in_split(split))} images')

In [ ]:
import yaml
data_yaml_path = os.path.join(base_path, 'data.yaml')
with open(data_yaml_path) as f:
    cfg = yaml.safe_load(f)
cfg['train'] = f'{base_path}/train/images'
cfg['val']   = f'{base_path}/valid/images'
cfg['test']  = f'{base_path}/test/images'
with open(data_yaml_path, 'w') as f:
    yaml.dump(cfg, f)
print('data.yaml updated. Classes:', cfg.get('names'))

In [ ]:
# ── Patch Boundary-Aware Mask Loss into ultralytics ────────────────────────
#
# Implementation strategy:
#   Ultralytics v8SegmentationLoss.calculate_segmentation_loss() computes
#   `loss_s = self.seg_loss_fn(pred_masks, gt_masks, ...)` using a BCELoss.
#   We monkey-patch the class to replace the loss calculation with
#   BCE + BoundaryIoU, keeping the rest of the loss pipeline untouched.
#
#   BoundaryIoU extracts the morphological boundary (dilation - erosion)
#   from both pred and gt masks, then computes IoU on those boundary regions.
#   This gives direct gradient signal toward sharp, aligned boundaries.

import torch
import torch.nn as nn
import torch.nn.functional as F
import ultralytics.utils.loss as ult_loss


class BoundaryAwareMaskLoss(nn.Module):
    """BCE + Boundary IoU loss for sharper segmentation boundaries."""

    def __init__(self, boundary_weight=0.5, kernel_size=3):
        super().__init__()
        self.bce            = nn.BCEWithLogitsLoss(reduction='none')
        self.boundary_weight = boundary_weight
        # MaxPool used to approximate morphological dilation
        pad = kernel_size // 2
        self.pool = nn.MaxPool2d(kernel_size, stride=1, padding=pad)

    def _extract_boundary(self, mask_prob):
        """Dilation - erosion (morphological boundary) in differentiable form."""
        # mask_prob: [B, H, W] float in [0,1]
        m = mask_prob.unsqueeze(1)     # [B,1,H,W]
        dilated = self.pool(m)
        eroded  = -self.pool(-m)
        return (dilated - eroded).squeeze(1)   # [B,H,W]

    def forward(self, pred_logits, gt_masks):
        """
        Args:
            pred_logits: [B, H, W]  raw logits
            gt_masks:    [B, H, W]  binary 0/1
        """
        # BCE on all pixels
        bce_loss = self.bce(pred_logits, gt_masks.float()).mean()

        # Boundary IoU
        pred_prob = pred_logits.sigmoid()
        pred_b = self._extract_boundary(pred_prob)
        gt_b   = self._extract_boundary(gt_masks.float())

        inter = (pred_b * gt_b).sum(dim=[1, 2])
        union = pred_b.sum(dim=[1, 2]) + gt_b.sum(dim=[1, 2]) - inter
        boundary_iou  = (inter + 1e-7) / (union + 1e-7)
        boundary_loss = (1 - boundary_iou).mean()

        return bce_loss + self.boundary_weight * boundary_loss


_boundary_loss_fn = BoundaryAwareMaskLoss(boundary_weight=0.5)


# Patch v8SegmentationLoss to use BoundaryAwareMaskLoss
# We wrap calculate_segmentation_loss so the change is localised.
_orig_seg_loss_cls = ult_loss.v8SegmentationLoss


def _patched_calculate_segmentation_loss(
    self, overlap_masks, masks, batch_idx, proto, pred_masks, imgsz, loss_coeff
):
    """
    Override: replaces BCE mask loss with BoundaryAwareMaskLoss.
    Called by v8SegmentationLoss.forward() for each image in the batch.
    """
    # -- Reconstruct predicted full mask (same logic as original) --
    # proto: [32, Hm, Wm], pred_masks: [N, 32]
    # We call the original method but capture the mask prediction tensors
    # by temporarily using a proxy BCE that records inputs.
    #
    # Simpler alternative: call original, then re-compute seg loss component.
    # However the original method returns a scalar and we cannot split BCE out.
    # Instead we monkey-patch self.seg_loss to our fn for this call.
    #
    orig_seg = self.seg_loss
    self.seg_loss = lambda pred, gt, *a, **kw: _boundary_loss_fn(pred, gt)
    result = _orig_seg_loss_method(self, overlap_masks, masks, batch_idx,
                                   proto, pred_masks, imgsz, loss_coeff)
    self.seg_loss = orig_seg
    return result


# Locate the original method name (varies slightly across ultralytics versions)
_orig_seg_loss_method = getattr(
    _orig_seg_loss_cls,
    'calculate_segmentation_loss',
    None,
)

if _orig_seg_loss_method is not None:
    _orig_seg_loss_cls.calculate_segmentation_loss = _patched_calculate_segmentation_loss
    print('BoundaryAwareMaskLoss patched into v8SegmentationLoss.calculate_segmentation_loss.')
else:
    # Fallback: directly override self.seg_loss during __init__
    _orig_init = _orig_seg_loss_cls.__init__
    def _new_init(self, *args, **kwargs):
        _orig_init(self, *args, **kwargs)
        self.seg_loss = lambda pred, gt, *a, **kw: _boundary_loss_fn(pred, gt)
    _orig_seg_loss_cls.__init__ = _new_init
    print('BoundaryAwareMaskLoss patched via __init__ override (fallback path).')

In [ ]:
# ── Register SimAM + CoordAtt in ultralytics namespace ─────────────────────
import torch
import torch.nn as nn
import ultralytics.nn.modules.conv as conv_module
import ultralytics.nn.modules as nn_modules
import ultralytics.nn.tasks as tasks_module


class SimAM(nn.Module):
    def __init__(self, c1=None, e_lambda=1e-4):
        super().__init__()
        self.e_lambda   = e_lambda
        self.activation = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()
        n = h * w - 1
        if n <= 0:
            return x
        x_mu  = x - x.mean(dim=[2, 3], keepdim=True)
        denom = 4 * (x_mu.pow(2).sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)
        y     = x_mu.pow(2) / denom + 0.5
        return x * self.activation(y)


class CoordAtt(nn.Module):
    def __init__(self, c1, c2=None, reduction=32):
        super().__init__()
        c2  = c1 if c2 is None else c2
        mip = max(8, c1 // reduction)
        self.conv1  = nn.Conv2d(c1, mip, 1, 1, 0)
        self.bn1    = nn.BatchNorm2d(mip)
        self.act    = nn.SiLU()
        self.conv_h = nn.Conv2d(mip, c2, 1, 1, 0)
        self.conv_w = nn.Conv2d(mip, c2, 1, 1, 0)
        self.proj   = nn.Conv2d(c1, c2, 1, 1, 0) if c1 != c2 else None

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        y   = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        a_h = self.conv_h(x_h).sigmoid()
        a_w = self.conv_w(x_w).sigmoid()
        if self.proj is not None:
            identity = self.proj(identity)
        return identity * a_h * a_w


for ns in [conv_module, nn_modules, tasks_module]:
    ns.SimAM    = SimAM
    ns.CoordAtt = CoordAtt
# __all__ is a tuple in ultralytics — convert to list before concatenating
nn_modules.__all__ = list(set(list(getattr(nn_modules, '__all__', [])) + ['SimAM', 'CoordAtt']))
print('SimAM + CoordAtt registered.')

In [ ]:
# ── Write YAML ─────────────────────────────────────────────────────────────
import ultralytics
from pathlib import Path

YAML_CONTENT = """
# YOLO11n-seg + SimAM + CA head — Boundary-Aware Mask Loss experiment (Nhom B, B2)
nc: 2
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, \"nearest\"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, \"nearest\"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]

  - [16, 1, CoordAtt, [256, 32]]
  - [23, 1, SimAM, []]

  - [19, 1, CoordAtt, [512, 32]]
  - [25, 1, SimAM, []]

  - [22, 1, CoordAtt, [1024, 32]]
  - [27, 1, SimAM, []]

  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]
""".strip()

ultralytics_root = Path(ultralytics.__file__).parent
yaml_dir  = ultralytics_root / 'cfg' / 'models' / '11'
yaml_dir.mkdir(parents=True, exist_ok=True)
yaml_path = yaml_dir / 'yolo11n-seg-simam-ca-boundary.yaml'
yaml_path.write_text(YAML_CONTENT)
print('YAML written to:', yaml_path)

In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────
import gc, time, shutil
import yaml, pandas as pd
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display

EXPERIMENT_ROOT = Path('/content/shrimp_boundary') if os.path.exists('/content') else Path('/kaggle/working/shrimp_boundary')
RUNS_DIR   = EXPERIMENT_ROOT / 'runs' / 'segment'
REPORT_DIR = EXPERIMENT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

COUNT_PENALTY_WEIGHT       = 0.05
DISEASE_MISS_PENALTY_WEIGHT = 0.15
HEALTHY_FP_PENALTY_WEIGHT  = 0.10
PREDICT_CONF_FOR_COUNT     = 0.25
TRAIN_IMGSZ = 640


def find_image_for_label(image_dir, label_name):
    stem = Path(label_name).stem
    for ext in IMAGE_EXTENSIONS:
        c = Path(image_dir) / f'{stem}{ext}'
        if c.exists(): return c
    return None

def write_data_yaml(dataset_dir, out_yaml):
    with open(data_yaml_path) as f: c = yaml.safe_load(f)
    c['train'] = str(Path(dataset_dir)/'train'/'images')
    c['val']   = str(Path(dataset_dir)/'valid'/'images')
    c['test']  = str(Path(dataset_dir)/'test'/'images')
    with open(out_yaml, 'w') as f: yaml.safe_dump(c, f, sort_keys=False)

def copy_dataset_for_experiment(exp_key):
    dst = EXPERIMENT_ROOT / exp_key / 'dataset'
    if dst.exists(): shutil.rmtree(dst)
    shutil.copytree(base_path, dst, ignore=shutil.ignore_patterns('*.cache'))
    remove_yolo_label_caches(dst)
    return dst

def copy_split_by_label_state(src, dst, split, want_labeled):
    si = Path(src)/split/'images'; sl = Path(src)/split/'labels'
    di = Path(dst)/split/'images'; dl = Path(dst)/split/'labels'
    di.mkdir(parents=True, exist_ok=True); dl.mkdir(parents=True, exist_ok=True)
    copied = 0
    for lp in sorted(sl.glob('*.txt')):
        lines = [l.strip() for l in lp.read_text().splitlines() if l.strip()]
        if bool(lines) != want_labeled: continue
        ip = find_image_for_label(si, lp.name)
        if ip is None: continue
        shutil.copy2(ip, di/ip.name); shutil.copy2(lp, dl/lp.name); copied += 1
    return copied

def make_eval_dataset(src, exp_key, state_name, want_labeled):
    dst = EXPERIMENT_ROOT/exp_key/f'dataset_{state_name}_eval'
    if dst.exists(): shutil.rmtree(dst)
    for sub in ['images','labels']: (dst/'train'/sub).mkdir(parents=True, exist_ok=True)
    for split in ['valid','test']: copy_split_by_label_state(src, dst, split, want_labeled)
    yp = dst/f'data_{state_name}.yaml'; write_data_yaml(dst, yp)
    return dst, yp

def metric_value(metrics, dotted_path, default=float('nan')):
    obj = metrics
    for part in dotted_path.split('.'):
        if not hasattr(obj, part): return default
        obj = getattr(obj, part)
    try: return float(obj)
    except: return default

def count_prediction_errors(model, images_dir, labels_dir, conf=PREDICT_CONF_FOR_COUNT):
    ips = sorted(p for p in Path(images_dir).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
    if not ips:
        nan = float('nan')
        return dict(images=0, gt_total=0, pred_mask_total=0, mask_count_mae=nan,
                    disease_images=0, disease_box_miss_rate=nan, disease_mask_miss_rate=nan)
    results = model.predict(source=[str(p) for p in ips], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    errs, di, dbm = [], 0, 0; gt_total = pred_total = 0
    for ip, res in zip(ips, results):
        lp = Path(labels_dir)/f'{ip.stem}.txt'
        gt = len([l for l in lp.read_text().splitlines() if l.strip()]) if lp.exists() else 0
        pm = len(res.masks) if res.masks is not None else 0
        pb = len(res.boxes) if res.boxes is not None else 0
        errs.append(abs(pm-gt)/max(1,gt))
        if gt > 0: di += 1; dbm += int(pb == 0)
        gt_total += gt; pred_total += pm
    return dict(images=len(ips), gt_total=gt_total, pred_mask_total=pred_total,
                mask_count_mae=sum(errs)/len(errs) if errs else float('nan'),
                disease_images=di,
                disease_box_miss_rate=dbm/di if di else float('nan'),
                disease_mask_miss_rate=float('nan'))

def healthy_false_positive_summary(model, images_dir, conf=PREDICT_CONF_FOR_COUNT):
    ips = sorted(p for p in Path(images_dir).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
    if not ips:
        return dict(healthy_images=0, healthy_mask_fp_rate=float('nan'),
                    healthy_fp_masks_per_image=float('nan'), healthy_avg_fp_confidence=0.0)
    results = model.predict(source=[str(p) for p in ips], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    with_fp = mt = 0; confs = []
    for res in results:
        mc = len(res.masks) if res.masks is not None else 0
        if mc > 0:
            with_fp += 1; mt += mc
            try: confs.extend(res.boxes.conf.cpu().tolist())
            except: pass
    n = len(ips)
    return dict(healthy_images=n, healthy_mask_fp_rate=with_fp/n,
                healthy_fp_masks_per_image=mt/n,
                healthy_avg_fp_confidence=sum(confs)/len(confs) if confs else 0.0)

def healthy_aware_score(lm50, ci, fpi):
    return lm50 - COUNT_PENALTY_WEIGHT*ci['mask_count_mae'] - DISEASE_MISS_PENALTY_WEIGHT*ci['disease_box_miss_rate'] - HEALTHY_FP_PENALTY_WEIGHT*fpi['healthy_mask_fp_rate']

def read_best_epoch_from_results(run_path):
    csv_path = Path(run_path)/'results.csv'
    if not csv_path.exists(): return {}
    df = pd.read_csv(csv_path); df.columns = [c.strip() for c in df.columns]
    mc = 'metrics/mAP50(M)'
    if mc not in df.columns: return {'epochs_ran': len(df)}
    bi = df[mc].idxmax(); best = df.iloc[bi]; last = df.iloc[-1]
    return dict(epochs_ran=len(df),
                best_epoch_by_mask_map50=int(best.get('epoch', bi+1)),
                best_val_mask_map50=float(best.get(mc, float('nan'))),
                last_train_seg_loss=float(last.get('train/seg_loss', float('nan'))),
                last_val_seg_loss=float(last.get('val/seg_loss', float('nan'))),
                seg_loss_gap_val_minus_train=float(last.get('val/seg_loss',0)-last.get('train/seg_loss',0)))

def disable_ultralytics_albumentations():
    try:
        import ultralytics.data.augment as aug
        class _NoOp:
            contains_spatial = False
            def __init__(self, *a, **kw): self.transform = None
            def __call__(self, labels): return labels
        aug.Albumentations = _NoOp
    except Exception as e:
        print('Albumentations patch skipped:', e)

print('Helper functions defined.')

In [ ]:
# ── Training — SimAM+CA + Boundary-Aware Mask Loss ─────────────────────────
CLEAN_TRAIN_ARGS = dict(
    auto_augment=None,
    erasing=0.0, mosaic=0.0, mixup=0.0, cutmix=0.0, copy_paste=0.0,
    fliplr=0.5, flipud=0.0,
    hsv_h=0.01, hsv_s=0.35, hsv_v=0.20,
    degrees=0.0, translate=0.05, scale=0.20,
    shear=0.0, perspective=0.0, multi_scale=0.0, bgr=0.0,
)

exp_key  = 'simam_ca_boundary'
run_name = f'yolo11n-seg_{exp_key}'

dataset_dir = copy_dataset_for_experiment(exp_key)
exp_yaml    = dataset_dir / 'data.yaml'
write_data_yaml(dataset_dir, exp_yaml)

labeled_eval_dir, labeled_eval_yaml = make_eval_dataset(dataset_dir, exp_key, 'labeled_only', True)
healthy_eval_dir, healthy_eval_yaml = make_eval_dataset(dataset_dir, exp_key, 'healthy_only', False)

disable_ultralytics_albumentations()

yolo = YOLO(str(yaml_path))
try:
    yolo.load('yolo11n-seg.pt')
except Exception as e:
    print('Pretrained load warning:', e)

start = time.time()
yolo.train(
    data=str(exp_yaml),
    task='segment',
    imgsz=TRAIN_IMGSZ,
    epochs=100,
    batch=16,
    patience=30,
    seed=42,
    deterministic=True,
    workers=0,
    project=str(RUNS_DIR),
    name=run_name,
    exist_ok=True,
    pretrained=True,
    plots=True,
    verbose=True,
    **CLEAN_TRAIN_ARGS,
)
train_time_min = (time.time() - start) / 60
print(f'Training finished in {train_time_min:.1f} min.')

In [ ]:
# ── Evaluation ─────────────────────────────────────────────────────────────
run_path   = RUNS_DIR / run_name
best_path  = run_path / 'weights' / 'best.pt'
best_model = YOLO(str(best_path))

full_val       = best_model.val(data=str(exp_yaml),          split='val',  imgsz=TRAIN_IMGSZ, plots=True,  verbose=False)
full_test      = best_model.val(data=str(exp_yaml),          split='test', imgsz=TRAIN_IMGSZ, plots=True,  verbose=False)
labeled_val    = best_model.val(data=str(labeled_eval_yaml), split='val',  imgsz=TRAIN_IMGSZ, plots=False, verbose=False)
labeled_test   = best_model.val(data=str(labeled_eval_yaml), split='test', imgsz=TRAIN_IMGSZ, plots=False, verbose=False)

labeled_test_count = count_prediction_errors(
    best_model, labeled_eval_dir/'test'/'images', labeled_eval_dir/'test'/'labels')
healthy_test_fp    = healthy_false_positive_summary(
    best_model, healthy_eval_dir/'test'/'images')

labeled_test_map50 = metric_value(labeled_test, 'seg.map50')
h_score = healthy_aware_score(labeled_test_map50, labeled_test_count, healthy_test_fp)

row = dict(
    experiment=exp_key,
    loss_variant='BCE + BoundaryIoU (weight=0.5)',
    train_time_min=round(train_time_min, 2),
    full_val_mask_map50=metric_value(full_val, 'seg.map50'),
    full_test_mask_map50=metric_value(full_test, 'seg.map50'),
    labeled_val_mask_map50=metric_value(labeled_val, 'seg.map50'),
    labeled_test_mask_map50=labeled_test_map50,
    labeled_test_mask_map50_95=metric_value(labeled_test, 'seg.map'),
    healthy_test_mask_fp_rate=healthy_test_fp['healthy_mask_fp_rate'],
    healthy_aware_score=h_score,
    **labeled_test_count,
)
row.update(read_best_epoch_from_results(run_path))

results_df = pd.DataFrame([row])
results_df.to_csv(REPORT_DIR / 'boundary_loss_results.csv', index=False)
display(results_df.T)

del best_model; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
results_png = run_path / 'results.png'
if results_png.exists():
    plt.figure(figsize=(14, 10))
    plt.imshow(mpimg.imread(results_png))
    plt.axis('off')
    plt.title('Boundary-Aware Mask Loss Training Curves')
    plt.show()